In [44]:
# importar la api key y el tenant
import os
import pandas as pd
import json
import requests

CONFIG_PATH = os.path.join("..","..","config.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

In [35]:
TENANT = config["Qlik_connection"]["Qlik_tenant"]
API_KEY= config["Qlik_connection"]["Qlik_4s"]

base = f"https://{TENANT}/api/v1"
headers = {"Authorization": f"Bearer {API_KEY}"}

In [39]:
resp = requests.get(f"{base}/apps", headers=headers)
print("Status code:", resp.status_code)
print("Response body:", resp.text)

Status code: 200
Response body: {"data":[{"attributes":{"id":"0d83ade6-9d52-4da1-8448-59341f421d08","name":"REDI LATAM","description":"","thumbnail":"/api/v1/apps/0d83ade6-9d52-4da1-8448-59341f421d08/media/files/1Monterrey.png","lastReloadTime":"2025-07-05T10:55:46.064Z","createdDate":"2025-02-06T19:05:30.048Z","modifiedDate":"2025-07-07T15:50:51.738Z","owner":"auth0|47f8a54afdebc603e7753184154e0aeaedfc1d3a970962d1355e2398671868ad","ownerId":"677dc3b540acfc79eec6dac3","dynamicColor":"","published":false,"publishTime":"","custom":{},"hasSectionAccess":true,"encrypted":true,"originAppId":"","isDirectQueryMode":false,"usage":"ANALYTICS","spaceId":"67a4fa985f80476eb2be74b5","_resourcetype":"app"},"privileges":[],"create":[]},{"attributes":{"id":"0e57ff25-9f70-4973-8134-78add8ada5a6","name":"4S Interno - Definición","description":"","thumbnail":"","lastReloadTime":"2025-07-05T07:11:23.243Z","createdDate":"2025-02-23T03:15:57.437Z","modifiedDate":"2025-07-05T07:11:49.547Z","owner":"auth0|47f

In [40]:
apps = resp.json().get("data", [])
print("Apps encontradas:", [a["attributes"]["name"] for a in apps])
print("IDs:", [a["attributes"]["id"] for a in apps])

Apps encontradas: ['REDI LATAM', '4S Interno - Definición', 'App Analyzer', '4S Interno - Auditoría', 'Reload Analyzer', '4S Interno - Auditoria Big Data', 'DEV - Investigación', '4S Corporativo', 'REDI - Testing Stage', 'Access Evaluator', '4S Interno - Gran Reporte de Verticalización', '4S Demo ELDI', 'REDI Database Extraction', 'REDI Mx - backup', 'REDI - Retail', 'Report Analyzer', 'TECH - Widgets', 'Answers Analyzer', 'Consumption report 2025-03-21', 'TECH - Camilo B.', 'test_core_data', 'REDI - Real Estate Data Insights', 'TECH Real Estate Data Insights - OV', 'REDI - Pruebas Tech', 'Script - 4S Interno - Inf Secundaria', '4S Interno - Opinion de Valor', 'Info Secundaria Loader', 'REDI Financiero', 'AI_chatbot_test', '4S Interno - Test Relacion Geografica inf secundaria', '4S Interno - Inf Sec (no usar)', 'Automation Analyzer', 'Test', 'Access Evaluator_Tenant ant.', '4S Interno - Inf Secundaria', 'TECH - Estudio Vertical (Pruebas)', 'TECH - REDI Interno', 'Consumption report 202

In [53]:
import websocket

APP_ID = config["Qlik_connection"]["AI_chatbot_test"]

url = f"wss://{TENANT}/app/{APP_ID}"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Sec-WebSocket-Protocol": "qlik.api"
}

def on_message(ws, message):
    print("Mensaje recibido:")
    print(message)
    response = json.loads(message)

    # OpenDoc -> obtiene el handle de documento (típicamente 1)
    if response.get("id") == 1:
        doc_handle = response["result"]["qReturn"]["qHandle"]

        # Crear objeto de sesión para listar hojas
        create_session_obj = {
            "jsonrpc": "2.0",
            "id": 2,
            "handle": doc_handle,
            "method": "CreateSessionObject",
            "params": {
                "qProp": {
                    "qInfo": {
                        "qType": "SheetList"
                    },
                    "qAppObjectListDef": {
                        "qType": "sheet",
                        "qData": {
                            "title": "/qMetaDef/title"
                        }
                    }
                }
            }
        }
        ws.send(json.dumps(create_session_obj))

    # Listar objetos del documento
    elif response.get("id") == 2:
        sheet_list_handle = response["result"]["qReturn"]["qHandle"]

        # Obtener layout del objeto de lista de hojas
        get_layout = {
            "jsonrpc": "2.0",
            "id": 3,
            "handle": sheet_list_handle,
            "method": "GetLayout",
            "params": {}
        }
        ws.send(json.dumps(get_layout))

    elif response.get("id") == 3:
        sheets = response["result"]["qLayout"]["qAppObjectList"]["qItems"]
        print(f"Hojas encontradas: {len(sheets)}")
        for s in sheets:
            print(f"- {s['qMeta']['title']} (ID: {s['qInfo']['qId']})")

# Función para abrir conexión
def on_open(ws):
    print("Conexión abierta con Qlik Engine")

    open_doc = {
        "jsonrpc": "2.0",
        "id": 1,
        "handle": -1,
        "method": "OpenDoc",
        "params": {
            "qDocName": APP_ID  # este es el ID real de la app
        }
    }
    ws.send(json.dumps(open_doc))

# Crear WebSocket
ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    header=[f"Authorization: Bearer {API_KEY}", "Sec-WebSocket-Protocol: qlik.api"]
)

ws.run_forever(sslopt={"cert_reqs": 0})

Conexión abierta con Qlik Engine
Mensaje recibido:
{"jsonrpc":"2.0","method":"OnConnected","params":{"qSessionState":"SESSION_ATTACHED"}}
Mensaje recibido:
{"jsonrpc":"2.0","id":1,"result":{"qReturn":{"qType":"Doc","qHandle":1,"qGenericId":"a51880f2-a531-4285-a74d-97fda1b0676f"}}}
Mensaje recibido:
{"jsonrpc":"2.0","id":2,"result":{"qReturn":{"qType":"GenericObject","qHandle":2,"qGenericType":"SheetList","qGenericId":"4ae82a8a-43ae-4bcf-96c5-c802e4aa4cc5"}},"change":[2]}
Mensaje recibido:
{"jsonrpc":"2.0","id":3,"result":{"qLayout":{"qInfo":{"qId":"4ae82a8a-43ae-4bcf-96c5-c802e4aa4cc5","qType":"SheetList"},"qMeta":{"privileges":["read","update","delete"]},"qSelectionInfo":{},"qAppObjectList":{"qItems":[{"qInfo":{"qId":"JaFDS","qType":"sheet"},"qMeta":{"title":"Insihght_advisor_test_v1","description":"","_resourcetype":"app.object","_objecttype":"sheet","id":"JaFDS","approved":false,"published":false,"owner":"retech@4srealestate.com","ownerId":"677dc3b540acfc79eec6dac3","createdDate":"2

False

## Al parecer ya logro crear una conexion via websocket a qlik y listar los objetos en una app...